In [1]:
from pathlib import Path
from fafbseg import flywire
from meshparty import trimesh_vtk
from multiprocessing import Pool
from tqdm import tqdm

import flybrains
import sys

REPO_ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(REPO_ROOT / "code"))

from get_mesh_neuron import *
from cell_groups import *
from color_utils import *
import pickle
import numpy as np 
import pandas as pd 
from make_projection_view_of_3d_rendering import *
import distinctipy

In [16]:
# Load data
Data_ROOT = Path.cwd().resolve().parents[1]

labial_cluster = pd.read_parquet(Data_ROOT/'data/labial_cluster_info_v783.parquet')
g2c = {}
g2colors = {}
for t in ['L1','L2','L3','water','high_salt']:#list(dict.fromkeys(labial_cluster.type)):
    g2c[t] = list(labial_cluster.flyid.values[labial_cluster.type==t])
    g2colors[t] = list(labial_cluster.color.values[labial_cluster.type==t])[0]
    

tarsal_cluster = pd.read_parquet(Data_ROOT/'data/atGRN_cluster_info_v783.parquet')

for t in ['a3','a6','a7','a9']:
    g2c[t] = list(tarsal_cluster.flyid.values[tarsal_cluster.type==t])
    g2colors[t] = list(tarsal_cluster.color.values[tarsal_cluster.type==t])[0]

    
g2c['TPN1'] = TPN1
g2c['MN9'] = MN9
g2c['av1a1'] = av1a1
g2colors['TPN1'] = '#5f5aa6'
g2colors['MN9'] = '#982598'
g2colors['av1a1'] = '#E4027F'



all_cells = list(np.concatenate(list(g2c.values())))

In [29]:
# pool = Pool(4)
# meshes_all = []
# for result in tqdm(pool.imap(load_mesh,all_cells),total=len(all_cells)):
#     meshes_all.append(result)
# pool.close()
# pool.join()

# # Load brain mesh
# available = flywire.get_neuropil_volumes(None)
# neuropil = flybrains.FLYWIRE.mesh
# mesh_ids = [x.id for x in meshes_all]
# meshes_all_navis = navis.NeuronList(meshes_all)
# Save
# with open("neurons.pkl", "wb") as f:
#     pickle.dump(meshes_all_navis, f)

# Load
with open("neurons.pkl", "rb") as f:
    meshes_all_navis = pickle.load(f)
meshes_all = [x for x in meshes_all_navis]

In [36]:

mesh_actors_dict = make_group_mesh_actors(
    meshes=meshes_all,
    g2c=g2c,
    g2color=g2colors,
    neuropil=flybrains.FLYWIRE.mesh,
    cell_opacity=1,
    neuropil_opacity=0.1
)

m = navis.NeuronList(meshes_all)
center = np.array([526232.30530339, 311428.70632942, 128201.97632289])

for g in ['L1','L2','L3','high_salt','water','a3','a6','a7','a9','TPN1','MN9']:
    mesh_actors = [*mesh_actors_dict[g],*mesh_actors_dict['neuropil']]
    
    make_projection_image(
        save_path=f'figures',
        actors_list=mesh_actors,
        center=center,
        projection_view='xy',
        backoff=700,
        parallelsacle=70000,
        do_save=False
    )


center = np.mean(np.vstack(neuropil.vertices),axis=0) + np.array([0,-19000,0])
g = 'av1a1'
mesh_actors = [*mesh_actors_dict[g],*mesh_actors_dict['neuropil']]

make_projection_image(
    save_path=f'figures',
    actors_list=mesh_actors,
    center=center,
    projection_view='xy',
    backoff=500,
    parallelsacle=150000,
    do_save=False
)

